In [1]:
"""
exp01 Step 01: Model Dataset Generation

Objective:

    - Join, for every decision bar, the feature rows (pipeline TSIND step02, TSSEG step03, TSCTX step04) with the barrier
      target row (pipeline TSBAR step05). Pairing: feature row at bar t <-> target row with decision_ts = t (entry at bar t+1).
    - Keep EVERY eligible decision row (no sampling here). Sampling for training is applied when the dataset is read
      (model_dataset.read_model_dataset_pdf), so it can change in the experiment config without regenerating this step.
    - The raw TSBAR cells are kept; labels and costs are parsed when the dataset is read.

No trial is logged here (no result is computed). The preview cells only show the exploration period (2005-2014).
Prerequisites: pipeline steps 01 to 05. Outputs: store04_experiments/exp01_minute_entry/step01_model_dataset_data/
(one file per session, model_dataset_YYYYMMDD.csv; about 5,400 files on a first run).
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_date_range_file_path_list, generate_func_data_date_list, get_missing_date_list
# IMPORT BARRIER LABEL FUNCTIONS
from so.features.barrier_labels import get_TSBAR_incomplete_date_list
# IMPORT MODEL DATASET FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import get_date_model_dataset_pdf, read_model_dataset_pdf, get_model_feature_col_str_list
# IMPORT TRIAL LOG FUNCTIONS (CONFIGURATION HASH ONLY: NOTHING IS LOGGED IN THIS STEP)
from so.core.trial_log import get_config_hash_str

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data")
print(f"Output folder:\t{output_path_str}")

Experiment:	exp01_minute_entry	Protocol:	1.0	Config Hash:	16493e598a
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/


In [4]:
# COLLECT THE TSBAR DATE RANGE (THE DATASET CANNOT EXTEND BEYOND THE TARGETS)
TSBAR_file_path_list = get_date_range_file_path_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR)
start_date_str = pd.to_datetime(TSBAR_file_path_list[0].split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d")
end_date_str = pd.to_datetime(TSBAR_file_path_list[-1].split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d")
# DISPLAY INFORMATION
print(f"TSBAR Date Range:\t{start_date_str} -> {end_date_str}\t({len(TSBAR_file_path_list):,} files)")

TSBAR Date Range:	2005-01-03 -> 2026-08-13	(5,436 files)


In [5]:
# DEFINE A SAMPLE DATE IN THE EXPLORATION PERIOD (THE LAST TSBAR DATE OF 2014)
sample_date_str = max(pd.to_datetime(f.split("_")[-1].replace(".csv", "")).strftime("%Y-%m-%d") for f in TSBAR_file_path_list
                      if pd.to_datetime(f.split("_")[-1].replace(".csv", "")) <= pd.Timestamp(config.EXPLORATION_END_DATE_STR))
# CALL FUNCTION TO GENERATE THE SAMPLE DATE MODEL DATASET
sample_date_dataset_pdf = get_date_model_dataset_pdf(sample_date_str)
# DISPLAY THE FEATURE NULL COUNTS (EARLY-SESSION INDICATORS SHOULD BE AVAILABLE BY 09:59)
feature_col_str_list = [col for col in config.FEATURE_REGISTRY_DICT if col in sample_date_dataset_pdf.columns]
print(f"Sample date:\t{sample_date_str}")
display(sample_date_dataset_pdf[feature_col_str_list].isna().sum().loc[lambda s: s > 0].reset_index().rename(columns={"index": "feature", 0: "null_count"}))
# PREVIEW DATAFRAME
sample_date_dataset_pdf

Sample date:	2014-12-31


,feature,null_count
0,ms_low_status,288
1,ms_high_status,321
2,ms_trend,118


,decision_ts,entry_ts,entry_price,horizon_complete,rr_ratio,open,high,low,close,volume,...,0.0071,0.0079,0.0089,0.0100,0.0112,0.0126,0.0141,0.0158,0.0178,0.0200
0,2014-12-31 09:59:00-05:00,2014-12-31 10:00:00-05:00,207.80,True,1.0,207.81,207.83,207.79,207.80,59300,...,279|206.325|SL,281|206.158|SL,345|205.951|SL,350|205.722|SL,352|205.473|SL,430|205.182|SL,433|204.87|SL,463|204.517|SL,756|204.101|SL,761|203.644|SL
1,2014-12-31 10:00:00-05:00,2014-12-31 10:01:00-05:00,207.92,True,1.0,207.80,207.92,207.80,207.92,108100,...,260|206.444|SL,278|206.277|SL,342|206.07|SL,345|205.841|SL,349|205.591|SL,428|205.3|SL,431|204.988|SL,459|204.635|SL,601|204.219|SL,758|203.762|SL
2,2014-12-31 10:01:00-05:00,2014-12-31 10:02:00-05:00,207.96,True,1.0,207.92,207.96,207.89,207.96,181000,...,258|206.483|SL,277|206.317|SL,279|206.109|SL,344|205.88|SL,348|205.631|SL,427|205.34|SL,430|205.028|SL,458|204.674|SL,600|204.258|SL,757|203.801|SL
3,2014-12-31 10:02:00-05:00,2014-12-31 10:03:00-05:00,207.89,True,1.0,207.96,207.97,207.89,207.89,163400,...,258|206.414|SL,278|206.248|SL,341|206.04|SL,343|205.811|SL,347|205.562|SL,426|205.271|SL,430|204.959|SL,460|204.605|SL,599|204.19|SL,756|203.732|SL
4,2014-12-31 10:03:00-05:00,2014-12-31 10:04:00-05:00,207.82,True,1.0,207.89,207.92,207.80,207.82,317500,...,275|206.344|SL,277|206.178|SL,340|205.97|SL,345|205.742|SL,348|205.492|SL,426|205.201|SL,429|204.89|SL,459|204.536|SL,747|204.121|SL,757|203.664|SL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,2014-12-31 15:54:00-05:00,2014-12-31 15:55:00-05:00,205.60,True,1.0,205.59,205.70,205.55,205.60,1163500,...,396|204.14|SL,402|203.976|SL,404|203.77|SL,413|203.544|SL,419|203.297|SL,425|203.009|SL,465|202.701|SL,495|202.352|SL,574|201.94|SL,709|201.488|SL
356,2014-12-31 15:55:00-05:00,2014-12-31 15:56:00-05:00,205.65,True,1.0,205.60,205.69,205.51,205.66,1702900,...,246|204.19|SL,401|204.025|SL,403|203.82|SL,411|203.594|SL,417|203.347|SL,424|203.059|SL,461|202.75|SL,494|202.401|SL,573|201.989|SL,696|201.537|SL
357,2014-12-31 15:56:00-05:00,2014-12-31 15:57:00-05:00,205.90,True,1.0,205.65,205.91,205.65,205.91,1217300,...,242|204.438|SL,245|204.273|SL,400|204.067|SL,402|203.841|SL,410|203.594|SL,417|203.306|SL,423|202.997|SL,464|202.647|SL,502|202.235|SL,573|201.782|SL
358,2014-12-31 15:57:00-05:00,2014-12-31 15:58:00-05:00,205.68,True,1.0,205.90,205.91,205.63,205.68,1168700,...,244|204.22|SL,399|204.055|SL,401|203.849|SL,404|203.623|SL,414|203.376|SL,422|203.088|SL,459|202.78|SL,492|202.43|SL,571|202.019|SL,629|201.566|SL


In [6]:
# COLLECT THE DATES WHOSE TARGETS ARE STILL INCOMPLETE (THEY CHANGE WHEN NEW DATA ARRIVES)
incomplete_date_list = get_TSBAR_incomplete_date_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, recent_file_count_in=30)
# DISPLAY INFORMATION
print(f"Incomplete TSBAR Dates:\t{len(incomplete_date_list)}")
# CALL FUNCTION TO REGENERATE THE 30 MOST RECENT DATES (OVERWRITE MODE; COVERS DATES WHOSE TARGETS WERE JUST REGENERATED IN STEP 05)
recent_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in TSBAR_file_path_list[-30:]]
generate_func_data_date_list(get_date_model_dataset_pdf, recent_date_list, output_path_str, "model_dataset", "W")

Incomplete TSBAR Dates:	9

Date:	2026-07-02	[1/30]
	⏳ Processing date...
	✅ Date:	'2026-07-02' Processed In 0.16 seconds (0.00 minutes)
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20260702.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20260702.csv' Has Been Saved!

Date:	2026-07-06	[2/30]
	⏳ Processing date...
	✅ Date:	'2026-07-06' Processed In 0.11 seconds (0.00 minutes)
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20260706.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/mode

,date,seconds_duration
0,2026-07-02,0.162919
1,2026-07-06,0.110760
2,2026-07-07,0.070715
3,2026-07-08,0.071866
4,2026-07-09,0.065186
5,2026-07-10,0.072225
6,2026-07-13,0.124631
7,2026-07-14,0.087045
8,2026-07-15,0.082464
9,2026-07-16,0.089263


In [7]:
# GET MISSING DATE LIST (EVERY DATE ON A FIRST RUN)
missing_date_list = get_missing_date_list(output_path_str, start_date_str, end_date_str)
# DISPLAY INFORMATION
print(f"Missing dates:\t{len(missing_date_list):,}")
# PREVIEW LIST
missing_date_list[:10]

Expected Date Count:	5,437
Existing Date Count:	30
Missing Date Count:	5,407
Missing dates:	5,407


[datetime.date(2005, 1, 3),
 datetime.date(2005, 1, 4),
 datetime.date(2005, 1, 5),
 datetime.date(2005, 1, 6),
 datetime.date(2005, 1, 7),
 datetime.date(2005, 1, 10),
 datetime.date(2005, 1, 11),
 datetime.date(2005, 1, 12),
 datetime.date(2005, 1, 13),
 datetime.date(2005, 1, 14)]

In [8]:
# CALL FUNCTION TO PROCESS DATE LIST (IGNORE MODE: EXISTING FILES ARE KEPT)
generate_func_data_date_list(get_date_model_dataset_pdf, missing_date_list, output_path_str, "model_dataset", "I")


Date:	2005-01-03	[1/5407]
	⏳ Processing date...
	✅ Date:	'2005-01-03' Processed In 0.08 seconds (0.00 minutes)
ℹ️ Saving In Ignore Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20050103.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20050103.csv' Has Been Saved!

Date:	2005-01-04	[2/5407]
	⏳ Processing date...
	✅ Date:	'2005-01-04' Processed In 0.10 seconds (0.00 minutes)
ℹ️ Saving In Ignore Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20050104.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step01_model_dataset_data/model_dataset_20050104.csv' Has 

,date,seconds_duration
0,2005-01-03,0.075379
1,2005-01-04,0.097116
2,2005-01-05,0.084136
3,2005-01-06,0.107435
4,2005-01-07,0.095886
...,...,...
5402,2026-06-25,0.059842
5403,2026-06-26,0.059024
5404,2026-06-29,0.101494
5405,2026-06-30,0.077107


In [9]:
# CALL FUNCTION TO READ THE LAST 3 MONTHS OF THE EXPLORATION PERIOD WITH THE TRAINING SAMPLING AND THE MODEL DELTA LABELS
preview_end_str = config.EXPLORATION_END_DATE_STR
preview_dataset_pdf = read_model_dataset_pdf(str((pd.to_datetime(preview_end_str) - pd.DateOffset(months=3)).date()), preview_end_str,
                                             exp_config.MODEL_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# DISPLAY INFORMATION
print(f"Sampled Rows:\t{len(preview_dataset_pdf):,}\t(mode: {exp_config.SAMPLING_MODE}, every {exp_config.SAMPLE_EVERY_N_MINUTES} minutes)")
print(f"Model Features ({len(get_model_feature_col_str_list(preview_dataset_pdf))}):\t{get_model_feature_col_str_list(preview_dataset_pdf)}")
# PREVIEW DATAFRAME
preview_dataset_pdf

Reading 65 model dataset files (2014-09-30 -> 2014-12-31, sampling: 'interval')
Sampled Rows:	1,536	(mode: interval, every 15 minutes)
Model Features (33):	['trend', 'segment', 'color', 'bot_wick_pct', 'body_pct', 'top_wick_pct', 'stoch_k', 'stoch_d', 'rsi', 'bb_pct', 'dc_pct', 'ms_minima_trend', 'ms_maxima_trend', 'ms_low_status', 'ms_high_status', 'ms_trend', 'minima_count', 'maxima_count', 'prev_seg_delta_pct', 'seg_cs_count', 'seg', 'minutes_since_open', 'minutes_to_close', 'day_of_week', 'overnight_gap_pct', 'prev_day_return_pct', 'prev_day_range_pct', 'prev_day_high_dist_pct', 'prev_day_low_dist_pct', 'intraday_return_pct', 'daily_volatility', 'relative_volume', 'ath_drawdown_pct']


,decision_ts,entry_ts,entry_price,horizon_complete,rr_ratio,open,high,low,close,volume,...,exit_bar_count_0.0079,exit_reason_0.0079,y_tp_0.0089,net_return_0.0089,exit_bar_count_0.0089,exit_reason_0.0089,y_tp_0.0100,net_return_0.0100,exit_bar_count_0.0100,exit_reason_0.0100
0,2014-09-30 09:59:00-04:00,2014-09-30 10:00:00-04:00,197.20,True,1.0,197.17,197.22,197.16,197.21,180700,...,377,SL,0.0,-0.009051,380,SL,0.0,-0.010152,395,SL
1,2014-09-30 10:14:00-04:00,2014-09-30 10:15:00-04:00,197.34,True,1.0,197.27,197.35,197.22,197.34,320800,...,354,SL,0.0,-0.009050,362,SL,0.0,-0.010149,379,SL
2,2014-09-30 10:29:00-04:00,2014-09-30 10:30:00-04:00,197.28,True,1.0,197.25,197.31,197.25,197.28,82300,...,340,SL,0.0,-0.009053,350,SL,0.0,-0.010153,365,SL
3,2014-09-30 10:44:00-04:00,2014-09-30 10:45:00-04:00,197.79,True,1.0,197.78,197.82,197.77,197.78,124800,...,321,SL,0.0,-0.009050,324,SL,0.0,-0.010152,324,SL
4,2014-09-30 10:59:00-04:00,2014-09-30 11:00:00-04:00,197.91,True,1.0,197.89,197.95,197.88,197.92,96600,...,305,SL,0.0,-0.009049,309,SL,0.0,-0.010151,309,SL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1531,2014-12-31 14:44:00-05:00,2014-12-31 14:45:00-05:00,206.37,True,1.0,206.26,206.40,206.25,206.38,286400,...,175,SL,0.0,-0.009046,178,SL,0.0,-0.010146,317,SL
1532,2014-12-31 14:59:00-05:00,2014-12-31 15:00:00-05:00,206.39,True,1.0,206.33,206.41,206.30,206.33,383300,...,160,SL,0.0,-0.009046,163,SL,0.0,-0.010145,302,SL
1533,2014-12-31 15:14:00-05:00,2014-12-31 15:15:00-05:00,206.42,True,1.0,206.39,206.46,206.37,206.43,442700,...,145,SL,0.0,-0.009044,148,SL,0.0,-0.010144,286,SL
1534,2014-12-31 15:29:00-05:00,2014-12-31 15:30:00-05:00,206.58,True,1.0,206.58,206.63,206.57,206.59,248400,...,103,SL,0.0,-0.009047,130,SL,0.0,-0.010146,133,SL
